### News

In [1]:
import sys, platform, subprocess
print("OS:     ", platform.system(), platform.release())
print("Python: ", sys.version.split()[0])
try:
    import torch
    print("Torch:  ", torch.__version__, "| CUDA GPU:", torch.cuda.is_available())
except ImportError:
    print("Torch:   not installed")
try:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No NVIDIA GPU found")
except FileNotFoundError:
    print("No NVIDIA GPU / driver found")

OS:      Linux 6.6.159+
Python:  3.13.16
Torch:   2.14.1+cu130 | CUDA GPU: True
GPU 0: Tesla T4 (UUID: GPU-a0b2a965-7712-bd17-bf60-8f2e38b9f151)



### Installation

In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Gemma-3!
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install msgspec
!pip install blake3
!pip install gguf
!pip install networkx==2.reinstall8.8 --force-

In [3]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!



> មូលហេតុ៖ cell ដំឡើងបាន `upgrade Pillow` ពី `11.3.0` ទៅ `13.0.0` ខណៈពេលដែល Colab កំពុង run។ Pillow ចាស់ត្រូវបានផ្ទុកក្នុង memory រួចហើយ។ បន្ទាត់ sys.modules.pop(x) if "PIL" in x នៅក្នុង cell ដំឡើង ព្យាយាមលុប PIL ចាស់ចេញ ប៉ុន្តែវាមិនអាចលុប C extension ដែលបានផ្ទុករួចហើយបានទេ។ ដូច្នេះ ផ្នែកថ្មី និងផ្នែកចាស់ត្រូវបានលាយគ្នា ហើយបង្ក error។



In [4]:
!pip install --force-reinstall --no-deps pillow==11.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 76.4 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0


In [5]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1113 01:02:08.605000 2130 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1113 01:02:08.654000 2130 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(
W1113 01:02:11.798000 2130 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1113 01:02:11.799000 2130 torch/_library/opaque_object.py:288]

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Gemma3 patching. Transformers: 5.22.2. vLLM: 0.33.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

We now add LoRA adapters so we only need to update a small amount of parameters!

In [6]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [7]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [9]:
from datasets import load_dataset
dataset = load_dataset('csv', data_files="dataasilida_cleaned.csv")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input,output', '__index_level_0__'],
        num_rows: 521
    })
})


In [10]:
import csv, re
import pandas as pd
from datasets import Dataset, DatasetDict

# 1. អានឯកសារ CSV ម្ដងមួយជួរ
rows = []
with open("dataasilida_cleaned.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)  # រំលងបន្ទាត់ header ដែលខូច "input,output"
    for r in reader:
        if len(r) == 2:                          # ជួរធម្មតា៖ សំណួរ, ចម្លើយ
            q, a = r
        elif len(r) == 1 and re.search(r"\?\s*,", r[0]):   # ជួរដែលសំណួរ និងចម្លើយជាប់គ្នា
            q, a = re.split(r"\?\s*,", r[0], maxsplit=1)
            q = q + "?"
        else:
            continue                             # រំលងជួរដែលខូចទាំងស្រុង
        rows.append({"input": q, "output": a})

# 2. សម្អាតអក្សរ
df = pd.DataFrame(rows)
for col in ["input", "output"]:
    df[col] = (df[col].str.replace("\u200b", "", regex=False)   # លុបដកឃ្លាមើលមិនឃើញ
                      .str.replace("\\n", "\n", regex=False)    # ប្ដូរ \n ទៅជាបន្ទាត់ថ្មីពិតប្រាកដ
                      .str.strip())
df = df[(df["input"] != "") & (df["output"] != "")].drop_duplicates()

# 3. បង្កើត dataset ដោយផ្ទាល់
dataset = DatasetDict({"train": Dataset.from_pandas(df, preserve_index=False)})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 517
    })
})


In [11]:
dataset['train']['input'][0]

'តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី'

In [12]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 517
    })
})


In [13]:
def formatting_prompts_func(examples):
    texts = []
    for instruction, output in zip(examples["input"], examples["output"]):
        messages = [
            {"role": "user",      "content": instruction.strip()},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages, add_generation_prompt=False, tokenize=False
        ).removeprefix("<bos>")      # ការពារកុំឲ្យមាន <bos><bos>
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset)
print(dataset['train']['text'][0])

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'text'],
        num_rows: 517
    })
})
<start_of_turn>user
តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី<end_of_turn>
<start_of_turn>model
ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការតាមរយៈ E-Commerce ម៉ាស៊ីនឆូតកាត (POS) ម៉ាស៊ីនអេធីអឹម (ATM) ដែលមានបិទស្លាកសញ្ញា ចេស៊ីប៊ី នៅទូទាំងប្រទេស និងពាសពេញពិភពលោក ដោយធ្វើការទូទាត់សាច់ប្រាក់ចេញពីគណនីរបស់ម្ចាស់ប័ណ្ណដែលមាននៅធនាគារ អេស៊ីលីដា។<end_of_turn>



In [ ]:
# dataset = load_dataset('csv', data_files="data_kh.csv")
# dataset = dataset.map(formatting_prompts_func, batched = True,)


In [16]:
dataset['train']['text'][0]

'<start_of_turn>user\nតើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី<end_of_turn>\n<start_of_turn>model\nប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការតាមរយៈ E-Commerce ម៉ាស៊ីនឆូតកាត (POS) ម៉ាស៊ីនអេធីអឹម (ATM) ដែលមានបិទស្លាកសញ្ញា ចេស៊ីប៊ី នៅទូទាំងប្រទេស និងពាសពេញពិភពលោក ដោយធ្វើការទូទាត់សាច់ប្រាក់ចេញពីគណនីរបស់ម្ចាស់ប័ណ្ណដែលមាននៅធនាគារ អេស៊ីលីដា។<end_of_turn>\n'

Let's see how row 100 looks like!

In [17]:
dataset['train']['input'][0]

'តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី'

In [18]:
dataset['train']['input'][0]

'តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី'

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [14]:
dataset['train']['output'][00]

'ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការតាមរយៈ E-Commerce ម៉ាស៊ីនឆូតកាត (POS) ម៉ាស៊ីនអេធីអឹម (ATM) ដែលមានបិទស្លាកសញ្ញា ចេស៊ីប៊ី នៅទូទាំងប្រទេស និងពាសពេញពិភពលោក ដោយធ្វើការទូទាត់សាច់ប្រាក់ចេញពីគណនីរបស់ម្ចាស់ប័ណ្ណដែលមាននៅធនាគារ អេស៊ីលីដា។'

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [20]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'text'],
        num_rows: 517
    })
})


In [ ]:
# from trl import SFTTrainer, SFTConfig
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     eval_dataset = None, # Can set up evaluation!
#     args = SFTConfig(
#         dataset_text_field = "text",
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4, # Use GA to mimic batch size!
#         warmup_steps = 5,
#         # num_train_epochs = 1, # Set this for 1 full training run.
#         max_steps = 60,
#         learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
#         logging_steps = 1,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         report_to = "none", # Use this for WandB etc
#     ),
# )

In [24]:
from trl import SFTTrainer, SFTConfig



> រៀបចំ trainer តែប៉ុណ្ណោះ វា មិនទាន់ train ទេ។ ការ train ពិតប្រាកដចាប់ផ្ដើមនៅ cell trainer.train() ពេលក្រោយ។



In [25]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    args = SFTConfig(
        dataset_text_field = "text",      # ផ្លាស់មកខាងក្នុង SFTConfig
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/517 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [26]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [27]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><start_of_turn>user\nដូចម្តេចដែលហៅថាសេវាបញ្ជាទូទាត់តាមរយៈ E-mail/Fax<end_of_turn>\n<start_of_turn>model\nសេវាបញ្ជាទូទាត់តាមរយៈ E-mail / Fax គឺជាប្រតិបត្តិការផ្ទេរប្រាក់ ឬដកប្រាក់ពីគណនី តាមការណែនាំរបស់ម្ចាស់គណនី ដែលមានជាមួយធនាគារ អេស៊ីលីដា ហើយបានផ្ញើ E-mail / Fax មកលោកនាយកប្រតិបត្តិការ ឬនាយក-នាយិកាសាខា នៃធនាគារ អេស៊ីលីដា ដើម្បីធ្វើប្រតិបត្តិការទូទាត់តាមបំណងរបស់ខ្លួន។<end_of_turn>\n'

Now let's print the masked out example - you should see only the answer is present:

In [28]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                សេវាបញ្ជាទូទាត់តាមរយៈ E-mail / Fax គឺជាប្រតិបត្តិការផ្ទេរប្រាក់ ឬដកប្រាក់ពីគណនី តាមការណែនាំរបស់ម្ចាស់គណនី ដែលមានជាមួយធនាគារ អេស៊ីលីដា ហើយបានផ្ញើ E-mail / Fax មកលោកនាយកប្រតិបត្តិការ ឬនាយក-នាយិកាសាខា នៃធនាគារ អេស៊ីលីដា ដើម្បីធ្វើប្រតិបត្តិការទូទាត់តាមបំណងរបស់ខ្លួន។<end_of_turn> '

In [29]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.381 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [30]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 106}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 517 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.359212
2,3.263336
3,3.284615
4,3.019245
5,2.873418
6,3.225585
7,2.439978
8,1.578408
9,2.039076
10,1.877523


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


In [31]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

296.3349 seconds used for training.
4.94 minutes used for training.
Peak reserved memory = 5.061 GB.
Peak reserved memory for training = 0.68 GB.
Peak reserved memory % of max memory = 34.752 %.
Peak reserved memory for training % of max memory = 4.669 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [32]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "ទៅមុខ១០មហើយបត់ឆ្វេង",
    }]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)
outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nទៅមុខ១០មហើយបត់ឆ្វេង<end_of_turn>\n<start_of_turn>model\nបត់ឆ្វេងទៅមុខ ១០០មេក។<end_of_turn>']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [33]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "ទៅមុខ១០ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

សូមទទួលកាតិកេតបថីជួសជិតបរជួញលេងបាត់។<end_of_turn>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [34]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

Unsloth: Restored added_tokens_decoder metadata in gemma-3/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3.


['gemma-3/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [35]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "បន្តទៅក្រោយ៩ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

អ៊ិលខ្នាអតិថិកា។<end_of_turn>


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [36]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [37]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

In [38]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [39]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )

In [41]:
from transformers import TextStreamer

def ask(question, max_new_tokens=256):
    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": question}],
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,   # ប្រាប់ម៉ូដែលថាដល់វេនវាឆ្លើយហើយ
        tokenize = True,
        return_tensors = "pt",
        return_dict = True,
    )
    print("សំណួរ៖", question)
    print("ចម្លើយ៖ ", end="")
    _ = model.generate(
        **inputs.to("cuda"),
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64,   # ការកំណត់ដែល Gemma 3 ណែនាំ
        streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True),
    )
    print("-" * 60)

In [42]:
ask("តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី")
ask("តើពេលបើកគណនីដំបូងត្រូវការតម្កល់ប្រាក់ឬទេ")

សំណួរ៖ តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី
ចម្លើយ៖ លីដា អេស៊ីលីដា ឥណពន្ធចេស៊ីប៊ី គឺជា ឯកសារឥណពន្ធ ដែលជារ៉ូបोटិកា ដែលមានឈ្មោះដើម ដែលបានចុះ និងចុះកិច្ចសន្យារិញ្ញវត្ថុ សំដៅ អេស៊ីលីដា ភីអិលស៊ី អាចធ្វើចរប្រតិបត្តិ ឥណពន្ធនៅតាមសក្តិដាពហុគ្រោងភារកិច្ច អ៊ីលីដាអេស៊ីហ្កាអ៉ីលីដាអាដអេល ធំ តូច និងមធ្យម អចិនបឋបត៍ ។ នេះជាលិខិតអចិនបឋបត៍ មួយប្រភេទ ផ្សេងទៀត ហើយអាចធ្វើជារ៉ូបोटិកា ក្នុងការប្រៀបថាតម្រាន់ ប្រើប្រាស់ កម្ពុជារូបអិនបាតចង់អមជាមួយ ឬ ប្រើប្រាស់ ភីអិលស៊ី ធានាតាមលក្ខខណ្ឌ។ ទីចម្បណ្ឌចារកិច្ចសញ្ជាយកិច្ចការអចិន
------------------------------------------------------------
សំណួរ៖ តើពេលបើកគណនីដំបូងត្រូវការតម្កល់ប្រាក់ឬទេ
ចម្លើយ៖ នៅពេលបើកគណនីដំបូង ហើយអ្នកបញ្ជូនប្រាក់មិនបានប្រាក់លើគណនី។ ដូច្នេះអ្នកបញ្ជូនប្រាក់មិនត្រូវបានគឹនកំហុសតាមគណនី ហើយប្រាក់ត្រូវបញ្ចូលបម្រុង។ ដូច្នេះបើកគណនីដំបូងសូមប្រាក់សម្រាប់ចុះ។
------------------------------------------------------------
